# Oracle Evaluation of Target Policies

**Date:** 2026-03-12

**Goal:** Compute ground-truth V^π (success rates) for the 4 trained diffusion target policies
(10, 50, 100, 200 demos) using 50 oracle rollouts each.

| Policy | Checkpoint Timestamp |
|--------|---------------------|
| 10 demos | `20260311115828` |
| 50 demos | `20260311134204` |
| 100 demos | `20260311135551` |
| 200 demos | `20260311141036` |

In [ ]:
import sys, os, time
import numpy as np
import json
from pathlib import Path

PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

from latent_sope.robomimic_interface.checkpoints import load_checkpoint
from latent_sope.eval.oracle import oracle_value, save_oracle_result

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ── Configuration ──
NUM_ROLLOUTS = 50
HORIZON = 60
GAMMA = 1.0

CKPT_BASE = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models"

POLICIES = {
    "10demos": CKPT_BASE / "lift_diffusion_10demos" / "20260311115828",
    "50demos": CKPT_BASE / "lift_diffusion_50demos" / "20260311134204",
    "100demos": CKPT_BASE / "lift_diffusion_100demos" / "20260311135551",
    "200demos": CKPT_BASE / "lift_diffusion_200demos" / "20260311141036",
}

RESULTS_DIR = PROJECT_ROOT / "results" / "2026-03-12"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for name, path in POLICIES.items():
    ckpt_file = path / "models" / "last.pth"
    print(f"{name:>10s}: {ckpt_file} — exists={ckpt_file.exists()}")

In [ ]:
# ── Run oracle evaluation for each policy ──
results = {}

for name, run_dir in POLICIES.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")
    
    t0 = time.time()
    ckpt = load_checkpoint(str(run_dir), ckpt_path="last.pth")
    
    result = oracle_value(
        ckpt,
        num_rollouts=NUM_ROLLOUTS,
        horizon=HORIZON,
        gamma=GAMMA,
        device=device,
        verbose=True,
        num_workers=0,
    )
    elapsed = time.time() - t0
    
    # Save per-policy oracle result
    oracle_json = run_dir / f"oracle_{NUM_ROLLOUTS}.json"
    save_oracle_result(oracle_json, result, policy_name=f"lift_diffusion_{name}")
    
    success_rate = float(np.mean(result.returns > 0))
    results[name] = {
        "mean_return": result.mean_return,
        "std_return": result.std_return,
        "success_rate": success_rate,
        "returns": result.returns.tolist(),
        "elapsed_sec": elapsed,
    }
    
    print(f"\n  V^π = {result.mean_return:.4f} ± {result.std_return:.4f}")
    print(f"  Success rate = {success_rate*100:.1f}%")
    print(f"  Time: {elapsed:.0f}s")

In [ ]:
# ── Summary table ──
print(f"\n{'='*70}")
print(f"ORACLE EVALUATION SUMMARY ({NUM_ROLLOUTS} rollouts each, horizon={HORIZON})")
print(f"{'='*70}")
print(f"{'Policy':>12s} | {'V^π':>10s} | {'± std':>10s} | {'Success%':>10s} | {'Time(s)':>8s}")
print(f"{'-'*12}-+-{'-'*10}-+-{'-'*10}-+-{'-'*10}-+-{'-'*8}")

for name in ["10demos", "50demos", "100demos", "200demos"]:
    r = results[name]
    print(f"{name:>12s} | {r['mean_return']:>10.4f} | {r['std_return']:>10.4f} | {r['success_rate']*100:>9.1f}% | {r['elapsed_sec']:>8.0f}")

# Save combined results
combined_path = RESULTS_DIR / "oracle_eval_target_policies.json"
with open(combined_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved combined results to {combined_path}")

In [ ]:
import matplotlib.pyplot as plt

names = ["10demos", "50demos", "100demos", "200demos"]
means = [results[n]["mean_return"] for n in names]
stds = [results[n]["std_return"] for n in names]
success_rates = [results[n]["success_rate"] * 100 for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart: V^π
ax1.bar(names, means, yerr=stds, capsize=5, color="steelblue", edgecolor="black")
ax1.set_ylabel("Oracle V^π (mean return)")
ax1.set_title("Oracle Value by Training Data Size")
ax1.grid(True, alpha=0.3, axis="y")

# Bar chart: Success rate
ax2.bar(names, success_rates, color="coral", edgecolor="black")
ax2.set_ylabel("Success Rate (%)")
ax2.set_title("Success Rate by Training Data Size")
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3, axis="y")

plt.suptitle(f"Target Policy Oracle Evaluation ({NUM_ROLLOUTS} rollouts each)", fontweight="bold")
plt.tight_layout()
plt.show()

print("Done!")